# D1.1 · From alert queue to loop operator

**Function D — Security Operations → The SOC Analyst & Detection Engineer**  ·  *AI for Security*

---

**Risk.** Supervising by re-reading everything the loop did.

**Control.** Know what the loop must escalate and sample the rest.

**This lab.** Supervise a triage loop by exception rather than by re-reading.

| | |
|---|---|
| Open-source tooling | Wazuh, OpenSearch |
| Open-weight models | GLM-4.6 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("D1.1"))

The alert queue becomes a loop you operate rather than a list you work. The skill that transfers is not triage speed — it is knowing which signal the loop is allowed to believe.

In [ ]:
from cybercommons import soc
import time

now = time.time()
events = [
    soc.Event(now,      "patch-agent", "http_get", "http://169.254.169.254/latest/meta-data/"),
    soc.Event(now + 1,  "patch-agent", "read_file", "/work/.env"),
    soc.Event(now + 2,  "dana",        "read_file", "/work/src/app.py"),
    soc.Event(now + 3,  "patch-agent", "run_shell", "", ok=False),
    soc.Event(now + 4,  "ci-runner",   "delete_repo", "repo/legacy"),
]
alerts = soc.run_rules(events, soc.default_rules())
for a in alerts:
    print(f"[{a.severity:8s}] {a.rule}")
    print(f"             actor={a.event.actor} target={a.event.target}")
    print(f"             → {a.response}")

Every alert carries a response. That is not documentation hygiene — an alert whose response is 'investigate' is an alert that will be closed without one.

In [ ]:
truth = {"patch-agent:http_get", "patch-agent:read_file", "ci-runner:delete_repo"}
print(soc.triage_quality(alerts, truth))

### Expect

Four alerts fire across three actors, each with a concrete response. Triage quality reports precision and recall plus alerts-per-true-positive.

### Your turn

Which of the four would you automate a response for today? The answer depends on precision, not severity — and that is the inversion this session is about.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/D1.1.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*